# Gdańsk Regional Championship: Day 2 team analysis

This notebook explores the 110 teams above the **Day 2 end** divider in the Gdańsk Regional Championship standings. It compares team compositions, groups teams connected by an exact similarity score, and demonstrates a damage matchup.

Keep `gdansk_day_2.json` beside this notebook and run the notebook with this folder as the working directory. The `pyvgc` package must be installed in the active Python environment.

In [17]:
import json
from pyvgc import (
    group_identical_teamplayers,
    pairwise_score_matrix,
)

with open("gdansk_day_2.json", encoding="utf-8") as data_file:
    standings_data = json.load(data_file)

teams = {
    entry["rank"]: {"player": entry["player"], "team": entry["team"]}
    for entry in standings_data[0:]
}
print(f"Loaded {len(teams)} teams ({sum(len(t['team']) for t in teams.values())} Pokémon sets).")
print("Standings source: https://standings.limitlessvgc.com/0044/standings")

Loaded 110 teams (660 Pokémon sets).
Standings source: https://standings.limitlessvgc.com/0044/standings


## Inspect the standings teams

Each record includes the player name and six sets containing Pokémon, item, ability, stat alignment (represented as nature by the package), and four moves.

In [18]:
for rank, entry in teams.items():
    roster = ", ".join(pokemon["name"] for pokemon in entry["team"])
    print(f"{rank:>3}. {entry['player']}: {roster}")

  1. Javier Varela: Gholdengo, Excadrill, Tyranitar, Staraptor, Sinistcha, Milotic
  2. Jude Gerard Lee Wei Cong: Garchomp, Sneasler, Eternal Flower Floette, Incineroar, Rillaboom, Kingambit
  3. Michael Kelsch: Kingambit, Sneasler, Salamence, Basculegion, Charizard, Whimsicott
  4. Ko Tsukide: Charizard, Sneasler, Kingambit, Garchomp, Basculegion, Whimsicott
  5. Liam Chatterjee: Eternal Flower Floette, Indeedee, Charizard, Glimmora, Garchomp, Whimsicott
  6. Nikola Zirdum: Rillaboom, Basculegion, Kingambit, Sneasler, Eternal Flower Floette, Salamence
  7. Dennis Komen: Basculegion, Kingambit, Whimsicott, Charizard, Salamence, Sneasler
  8. Alex Soto: Sinistcha, Eternal Flower Floette, Kingambit, Incineroar, Sneasler, Delphox
  9. Naoya Takasago: Garchomp, Basculegion, Sneasler, Kingambit, Charizard, Whimsicott
 10. Víctor Medina: Incineroar, Whimsicott, Garchomp, Charizard, Glimmora, Milotic
 11. Sebastian Liu Li: Whimsicott, Basculegion, Kingambit, Charizard, Garchomp, Eternal Flowe

## Compare team similarity

The score starts at 102 for two identical six-Pokémon teams.

More documentation about this function in the future.

A **higher score means more similar teams**. This implementation accepts the raw website team dictionaries, so it can also analyze sets that are outside the package's current M-C legality snapshot.

In [19]:
score_matrix = pairwise_score_matrix(teams)
column_width = max(4, len(str(max(teams))) + 1)

print("Rows and columns are standings ranks; identical teams score 102.")
print(" " * column_width + "".join(f"{rank:>{column_width}}" for rank in teams))
for rank, scores in score_matrix.items():
    print(f"{rank:>{column_width}}" + "".join(
        f"{scores[other_rank]:>{column_width}}" for other_rank in teams
    ))
print(f"Validated a symmetric {len(teams)} × {len(teams)} score matrix.")

Rows and columns are standings ranks; identical teams score 102.
       1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107 108 109 110
   1 102   0   0   0   0   0   0  15   0  13   0  34  34   0   0   0   0   0  16   0   0   0   0   0 101   0   0   0   0   0   0  17   0  17   0   0   0   0   0   0   0   0  16   0   0   0  33   0   0   0  16  48  34  34   0   0   0   0  15   0  67   0  13  51   0 100   0   0   0   0  34   0  17   0  47   0   0   0   0   0   0  17   0   0   0   0   0   0   0  14   0  33  34   0   0  34   0   0  16   0   0  96  15   0   0   0  15  47 101   0
   2   0 102  25  39  29  63  25  63  39  29 

## Find score-linked groups

A pair is included only when its score equals the selected score. Groups are connected components: if A matches B and B matches C at that score, all three players are grouped together even if A and C are not directly paired.

## Exact-score pairs and player groups

Team comparison, pairwise matrix generation, exact-score pair lookup, and transitive player grouping are provided by the public `pyvgc` API. Grouping follows connected components: a chain of matching pairs joins its members even if not every pair within a group directly matches.

In [20]:
target_score_pairs = 102

score_groups = group_identical_teamplayers(score_matrix)
print("\nGroups of players with identical teams (score 102):")
for group in score_groups:
    members = ", ".join(
        f"{rank} ({teams[rank]['player']})" for rank in group
    )
    print(members)


Groups of players with identical teams (score 102):
3 (Michael Kelsch), 7 (Dennis Komen)
4 (Ko Tsukide), 9 (Naoya Takasago), 21 (Amin Lehna), 29 (Michał Kwiatkowski), 33 (Yannick Otting), 49 (Justin Cerioni), 74 (Aleksandra Ćwikiel), 80 (Alessandro Pietro Tucci)
12 (Flavio Del Pidio), 13 (Francesco Pio Pero), 53 (Louis Fontvieille)
15 (Lorenzo Colombo), 56 (Daniele Kaygorodova)
16 (Marco Hemantha Kaludura Silva), 24 (Federico Camporesi)
18 (Alberto Noguerol), 40 (Juan Manuel Odriozola)
20 (Fabian Kuhl), 104 (Maurice Uteg)
26 (Nikolaj Høj Nielsen), 88 (Frederik Nielsen)
36 (Federico Masu), 69 (Stefano De Maria)
38 (Danilo Modugno), 48 (Rob Akershoek), 70 (Sebastian Wnuczek), 89 (Sergio Pascual Pino)
54 (Khaled Abusaada), 96 (Davide Aondio)
